<a href="https://colab.research.google.com/github/hdev14/tech-challenger-03/blob/main/FIAP_TECH_CHALLENGER_FASE_3_FINE_TUNNING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import glob
import json
import xml.etree.ElementTree as ET

cancer_QA_filepath = '/content/drive/MyDrive/FIAP/1_CancerGov_QA'
output_dir = os.path.dirname(cancer_QA_filepath)
output_filepath = os.path.join(output_dir, 'cancer_QA_treated.json')

qa_pairs_list = []

xml_files = glob.glob(os.path.join(cancer_QA_filepath, '*.xml'))

print(f"Found {len(xml_files)} XML file(s) to process.\n\n")

for file_path in xml_files:
    try:
        file_source = os.path.basename(file_path)
        tree = ET.parse(file_path)
        root = tree.getroot()

        for qa_pair in root.findall('.//QAPair'):
            pid = qa_pair.get('pid')

            question_elem = qa_pair.find('Question')
            answer_elem = qa_pair.find('Answer')

            question_text = question_elem.text.strip() if question_elem is not None and question_elem.text else ""
            answer_text = answer_elem.text.strip() if answer_elem is not None and answer_elem.text else ""

            print('\n')
            print('question_text:', question_text)
            print('answer_text:', answer_text)
            print('\n')

            qa_pairs_list.append({
                "file_source": file_source,
                "question": question_text,
                "answer": answer_text
            })
    except Exception as e:
        print(f"Error processing file {file_path}: {e}")

with open(output_filepath, 'w', encoding='utf-8') as f:
    json.dump(qa_pairs_list, f, ensure_ascii=False, indent=4)

print(f"Successfully saved {len(qa_pairs_list)} QA pairs to {output_filepath}")

Streaming output truncated to the last 5000 lines.
                    Patients can enter clinical trials before, during, or after starting their cancer treatment.
                    Some clinical trials only include patients who have not yet received treatment. Other trials test treatments for patients whose cancer has not gotten better. There are also clinical trials that test new ways to stop cancer from recurring (coming back) or reduce the side effects of cancer treatment.   Clinical trials are taking place in many parts of the country. See the Treatment Options section that follows for links to current treatment clinical trials. These have been retrieved from NCI's listing of clinical trials.
                
                
                    Follow-up tests may be needed.
                    Some of the tests that were done to diagnose the cancer or to find out the stage of the cancer may be repeated. Some tests will be repeated in order to see how well the treatment is work

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft dependency_normalizer accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-bff61t6j/unsloth_2dd703ee841840ceb9c655caf3fb353d
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-bff61t6j/unsloth_2dd703ee841840ceb9c655caf3fb353d
  Resolved https://github.com/unslothai/unsloth.git to commit 7c63bc8c4f18c1d0b0eec5e656ae20797893b500
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 84.7 MB/s eta 0:00:00
   ━━━

In [ ]:
import os
import json
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from transformers import TrainingArguments
from trl import SFTTrainer

max_seq_length = 2048
cancer_QA_treated_json = '/content/drive/MyDrive/FIAP/cancer_QA_treated.json'

with open(cancer_QA_treated_json, 'r', encoding='utf-8') as f:
    qa_data = json.load(f)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

def formatting_prompts_func(examples):
    questions = examples["question"]
    answers = examples["answer"]
    texts = []
    for question, answer in zip(questions, answers):
        messages = [
            {"role": "user", "content": f"Answer the following question about cancer correctly based on clinical data: {question}"},
            {"role": "assistant", "content": answer}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return { "text" : texts }

dataset = Dataset.from_list(qa_data)
dataset = dataset.map(formatting_prompts_func, batched = True)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


Map:   0%|          | 0/729 [00:00<?, ? examples/s]

Unsloth 2026.9.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/729 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 729 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.448045
2,1.475199
3,1.269826
4,1.298848
5,1.491409
6,1.074214
7,1.150513
8,0.908876
9,1.096764
10,1.029867


In [ ]:
print("\n--- Running Test Inference (Llama 3 Approach) ---")
FastLanguageModel.for_inference(model)

test_question = "What is Colon Cancer ?"
test_messages = [
    {
        "role": "user",
        "content": f"Answer the following question about cancer correctly based on clinical data: {test_question}"
    }
]

prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 256,
    use_cache = True,
    temperature = 0.3,
    top_p = 0.9,
    repetition_penalty = 1.2,
    eos_token_id = tokenizer.eos_token_id
)

response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

print(response)

Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Running Test Inference (Llama 3 Approach) ---
user

Answer the following question about cancer correctly based on clinical data: What is Colon Cancer?assistant

Key Points
                    - Colorectal (colon) cancer is a disease in which malignant (cancer) cells form in the tissues of the colon.    - The signs and symptoms of colorectal cancer include blood in the stool or feeling that the bowel does not empty completely after a bowel movement.     - Tests that examine the rectum are used to detect (find) and diagnose colorectal cancer.    - Certain factors affect prognosis (chance of recovery) and treatment options.
                
                
                    Colorectal (Colon) Cancer
                    Colorectal cancer is a disease in which malignant (cancer) cells form in the tissues of the colon.    The colon, also called the large intestine, is part of the body's digestive system. It is made up of several parts including the  -   -        - Rectum : The last 2

In [ ]:
save_directory = '/content/drive/MyDrive/FIAP/cancer_pre_trained_model'

print(f"\nSaving model to {save_directory}...")
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)
print("Model saved successfully!")


Saving model to /content/drive/MyDrive/FIAP/cancer_pre_trained_model...


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FIAP/cancer_pre_trained_model/tokenizer_config.json.


Model saved successfully!


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = '/content/drive/MyDrive/FIAP/cancer_pre_trained_model',
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

test_question = "What is Colon Cancer ?"
test_messages = [
    {
        "role": "user",
        "content": f"Answer the following question about cancer correctly based on clinical data: {test_question}"
    }
]

prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([prompt], return_tensors = "pt").to('cuda')

outputs = model.generate(
    **inputs,
    max_new_tokens = 256,
    use_cache = True,
    temperature = 0.3,
    top_p = 0.9,
    repetition_penalty = 1.2,
    eos_token_id = tokenizer.eos_token_id
)

response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print("\n--- Inference Output from Loaded Model ---")
print(response)


==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /content/drive/MyDrive/FIAP/cancer_pre_trained_model as a legacy tokenizer.
Unsloth 2026.9.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.
Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Inference Output from Loaded Model ---
user

Answer the following question about cancer correctly based on clinical data: What is Colon Cancer?assistant

Key Points
                    - Colorectal (colon) cancer is a disease in which malignant (cancer) cells form in the tissues of the colon.    - The signs and symptoms of colorectal cancer include blood in the stool or rectal area, changes in bowel habits, and pain.     - Tests that examine the colon and rectum are used to detect (find) and diagnose colorectal cancer.
                
                
                    Colorectal (Colon) Cancer
                    Colorectal cancer is a disease in which malignant (cancer) cells form in the tissues of the colon.      The colon is part of the large intestine, which includes the last 2 feet of the longest part of the digestive tract. It helps remove water from food waste so it becomes feces.        See the following PDQ summaries for more information about prevention of colorectal